# QuickBite Express - Crisis Recovery Analysis

## Notebook 1: Data Overview

This notebook covers data loading, structure review, data types, 
null value checks, and phase assignment across all 8 datasets.

The data spans January 2025 to September 2025. 
The crisis period began in June 2025 following a food safety 
incident and a week-long delivery outage.


In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

## Data Loading

Loading all 8 datasets from the data folder.

In [6]:
BASE_PATH = "../data/"

orders      = pd.read_csv(BASE_PATH + "fact_orders.csv")
order_items = pd.read_csv(BASE_PATH + "fact_order_items.csv")
ratings     = pd.read_csv(BASE_PATH + "fact_ratings.csv")
delivery    = pd.read_csv(BASE_PATH + "fact_delivery_performance.csv")
customers   = pd.read_csv(BASE_PATH + "dim_customer.csv")
restaurants = pd.read_csv(BASE_PATH + "dim_restaurant.csv")
partners    = pd.read_csv(BASE_PATH + "dim_delivery_partner_.csv")
menu        = pd.read_csv(BASE_PATH + "dim_menu_item.csv")

print("All datasets loaded successfully")

All datasets loaded successfully


## Dataset Overview

Reviewing the shape and column count of each dataset.

In [7]:
datasets = {
    'fact_orders'           : orders,
    'fact_order_items'      : order_items,
    'fact_ratings'          : ratings,
    'fact_delivery'         : delivery,
    'dim_customer'          : customers,
    'dim_restaurant'        : restaurants,
    'dim_delivery_partner'  : partners,
    'dim_menu_item'         : menu
}

for name, df in datasets.items():
    print(f"{name} | Rows: {len(df):,} | Columns: {len(df.columns)}")

fact_orders | Rows: 149,166 | Columns: 11
fact_order_items | Rows: 342,994 | Columns: 8
fact_ratings | Rows: 68,842 | Columns: 7
fact_delivery | Rows: 149,166 | Columns: 4
dim_customer | Rows: 107,776 | Columns: 4
dim_restaurant | Rows: 19,995 | Columns: 7
dim_delivery_partner | Rows: 15,000 | Columns: 7
dim_menu_item | Rows: 342,671 | Columns: 6


## Null Value Check

Checking for missing values across all datasets before analysis.

In [8]:
for name, df in datasets.items():
    nulls = df.isnull().sum().sum()
    print(f"{name} | Null values: {nulls}")

fact_orders | Null values: 5635
fact_order_items | Null values: 0
fact_ratings | Null values: 119
fact_delivery | Null values: 0
dim_customer | Null values: 0
dim_restaurant | Null values: 0
dim_delivery_partner | Null values: 0
dim_menu_item | Null values: 0


## Data Types Review

Reviewing column data types for each dataset to identify 
any columns that need type conversion before analysis.

In [9]:
for name, df in datasets.items():
    print(f"\n{name}")
    print(df.dtypes)


fact_orders
order_id                   str
customer_id                str
restaurant_id              str
delivery_partner_id        str
order_timestamp            str
subtotal_amount        float64
discount_amount        float64
delivery_fee           float64
total_amount           float64
is_cod                     str
is_cancelled               str
dtype: object

fact_order_items
order_id             str
item_id              str
menu_item_id         str
restaurant_id        str
quantity           int64
unit_price       float64
item_discount    float64
line_total       float64
dtype: object

fact_ratings
order_id                str
customer_id             str
restaurant_id           str
rating              float64
review_text             str
review_timestamp        str
sentiment_score     float64
dtype: object

fact_delivery
order_id                           str
actual_delivery_time_mins        int64
expected_delivery_time_mins      int64
distance_km                    float64
dtype

## Date Parsing and Phase Assignment

The order_timestamp column is converted to datetime format.
Orders are then categorized into three business phases based 
on the timeline of events.

Pre-Crisis  : January 2025 to May 2025
Crisis      : June 2025 to July 2025
Recovery    : August 2025 to September 2025

In [10]:
orders['order_timestamp'] = pd.to_datetime(orders['order_timestamp'])

def assign_phase(ts):
    if ts < pd.Timestamp('2025-06-01'):
        return 'Pre-Crisis'
    elif ts < pd.Timestamp('2025-08-01'):
        return 'Crisis'
    else:
        return 'Recovery'

orders['phase']             = orders['order_timestamp'].apply(assign_phase)
orders['month']             = orders['order_timestamp'].dt.to_period('M')
orders['is_cancelled_bool'] = orders['is_cancelled'] == 'Y'

print(orders['phase'].value_counts())

phase
Pre-Crisis    113806
Crisis         18111
Recovery       17249
Name: count, dtype: int64


## Date Range and Monthly Order Count

Verifying the date range of the dataset and reviewing 
order volume month by month.

In [11]:
print("Date Range")
print("From :", orders['order_timestamp'].min().date())
print("To   :", orders['order_timestamp'].max().date())

print("\nMonthly Order Count")
print(orders.groupby('month')['order_id'].count().to_string())

Date Range
From : 2025-01-01
To   : 2025-09-30

Monthly Order Count
month
2025-01    23539
2025-02    22667
2025-03    23543
2025-04    21466
2025-05    22591
2025-06     9293
2025-07     8818
2025-08     8555
2025-09     8694
Freq: M


## Sample Data Preview

Viewing first few rows of the primary fact tables 
to understand structure and values.

In [12]:
print("fact_orders sample")
print(orders.head(3).to_string())

print("\nfact_ratings sample")
print(ratings.head(3).to_string())

print("\nfact_delivery sample")
print(delivery.head(3).to_string())

fact_orders sample
          order_id customer_id restaurant_id delivery_partner_id     order_timestamp  subtotal_amount  discount_amount  delivery_fee  total_amount is_cod is_cancelled       phase    month  is_cancelled_bool
0  ORD202501023439  CUST181110     REST08622             DP05541 2025-01-01 12:00:00           471.62            35.44         30.56        466.74      N            N  Pre-Crisis  2025-01              False
1  ORD202501012051  CUST025572     REST02383             DP08091 2025-01-01 12:00:00           255.68             0.00         27.45        283.13      Y            N  Pre-Crisis  2025-01              False
2  ORD202501019281  CUST179306     REST14069             DP02021 2025-01-01 12:00:00           428.38             0.00         26.23        454.61      N            N  Pre-Crisis  2025-01              False

fact_ratings sample
          order_id customer_id restaurant_id  rating          review_text  review_timestamp  sentiment_score
0  ORD202501023439  CUS

## Null Value Detail

Identifying which specific columns contain null values 
in fact_orders and fact_ratings.

In [13]:
print("fact_orders null breakdown")
print(orders.isnull().sum())

print("\nfact_ratings null breakdown")
print(ratings.isnull().sum())

fact_orders null breakdown
order_id                  0
customer_id               0
restaurant_id             0
delivery_partner_id    5635
order_timestamp           0
subtotal_amount           0
discount_amount           0
delivery_fee              0
total_amount              0
is_cod                    0
is_cancelled              0
phase                     0
month                     0
is_cancelled_bool         0
dtype: int64

fact_ratings null breakdown
order_id            17
customer_id         17
restaurant_id       17
rating              17
review_text         17
review_timestamp    17
sentiment_score     17
dtype: int64


## Handling Null Values

delivery_partner_id has 5,635 null values in fact_orders. 
These rows are retained as the order data itself is complete. 
The missing partner ID likely indicates unassigned or cancelled orders.

fact_ratings has 17 fully blank rows across all columns. 
These rows are dropped as they contain no usable information.

In [14]:
# Drop 17 fully null rows from fact_ratings
ratings = ratings.dropna()

print("fact_ratings rows after dropping nulls:", len(ratings))
print("fact_orders delivery_partner_id nulls retained:", 
      orders['delivery_partner_id'].isnull().sum())

fact_ratings rows after dropping nulls: 68825
fact_orders delivery_partner_id nulls retained: 5635


## Summary

All 8 datasets were loaded and reviewed successfully.
The data covers January 2025 to September 2025 with 149,166 total orders.

Orders are categorized into three phases:
Pre-Crisis (Jan-May): 113,806 orders
Crisis (Jun-Jul): 18,111 orders  
Recovery (Aug-Sep): 17,249 orders

The sharp decline from approximately 22,600 orders per month 
in the pre-crisis period to 8,800 orders per month during 
the crisis indicates a severe business impact beginning June 2025.

Data is clean and ready for primary analysis.